Create the Gold Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS openskyreal.gold;

Read Silver Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/silver/flights"

silver_df2 = (
    spark.read
    .format("delta")
    .option("header","true")
    .option("inferSchema","true")
    .load(silver_path1)
)

display(silver_df2)

Country Traffic 

Use for:

Active flights per country
Top 10 countries

In [0]:
from pyspark.sql.functions import *

country_traffic_df1 = (
    silver_df2
    .groupBy("origin_country")
    .agg(
        countDistinct("icao24").alias("active_flights")
    )
    .orderBy(desc("active_flights"))
)

display(country_traffic_df1)

In [0]:
country_traffic_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.gold.country_traffic")

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


(
country_traffic_df1.write
.format("csv")
.mode("overwrite")
.option("mergeSchema","true")
.save(gold_path)
)

Speed Analytics 

Use for:

Average speed
Fastest aircraft
Speed distribution

In [0]:
speed_analysis_df1 = (
    silver_df2
    .groupBy("origin_country")
    .agg(
        avg("speed_kmh").alias("average_speed"),
        max("speed_kmh").alias("fastest_speed")
    )
)

display(speed_analysis_df1)

In [0]:
speed_analysis_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.gold.speed_analysis")

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


(
speed_analysis_df1.write
.format("csv")
.mode("overwrite")
.option("mergeSchema","true")
.save(gold_path)
)

Flight Direction Analysis 

Use for:

Direction heatmap
Air corridor detection

In [0]:
direction_analysis_df1 = (
    silver_df2
    .groupBy("direction")
    .agg(
        count("*").alias("flight_count")
    )
)

display(direction_analysis_df1)

In [0]:
direction_analysis_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.gold.direction_analysis")

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


(
direction_analysis_df1.write
.format("csv")
.mode("overwrite")
.option("mergeSchema","true")
.save(gold_path)
)

Aircraft Anomaly 

Use:

Sudden speed changes
Ground aircraft movement

In [0]:
anomaly_summary_df1 = (
    silver_df2
    .groupBy("origin_country")
    .agg(
        sum("speed_anomaly_flag").alias("speed_anomalies"),
        sum("ground_movement_flag").alias("ground_movements")
    )
)

display(anomaly_summary_df1)

In [0]:
anomaly_summary_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.gold.anomaly_summary")

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


(
anomaly_summary_df1.write
.format("csv")
.mode("overwrite")
.option("mergeSchema","true")
.save(gold_path)
)

Airspace Congestion 

Use:

Aircraft density
Busy airspace detection

In [0]:
airspace_congestion_df1 = (
    silver_df2
    .groupBy("region")
    .agg(
        countDistinct("icao24")
        .alias("aircraft_density")
    )
)

display(airspace_congestion_df1)

In [0]:
airspace_congestion_df1.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("openskyreal.gold.airspace_congestion")

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


(
airspace_congestion_df1.write
.format("csv")
.mode("overwrite")
.option("mergeSchema","true")
.save(gold_path)
)

In [0]:
%sql
SHOW TABLES IN openskyreal.gold;

In [0]:
%sql
SELECT *
FROM openskyreal.gold.country_traffic;